# Speaker Feedback System - End-to-End Pipeline

This notebook runs speech, slide, and visual analyses to produce:
- per-slide recommendations (text + tables/figures captured from OCR)
- overall presentation feedback
- JSON and Markdown outputs for report/PDF generation


## 1. Setup and Imports
Load project modules and dependencies once so every later cell can reuse them.


In [ ]:
import os
import json
import gc
import logging
from pathlib import Path
from collections import defaultdict
from typing import Any, Dict, List, Optional

import numpy as np
import cv2
import torch
import imageio_ffmpeg

from agents.tools.tool_registry import nemo_react_recommendation_tool
from agents.tools.face_cache_tools import build_face_cache_tool
from agents.tools.clothing_tool import clothing_analysis_tool
from agents.tools.emotion_tool import emotion_analysis_tool
from agents.tools.gaze_tool import gaze_analysis_tool
from agents.tools.gesture_tool import gestures_analysis_tool
from agents.tools.recommendation_tool import run_nemo_react_recommendations
from agents.tools.speech_analysis_tool import analyze_speech_tool
from agents.tools.slide_ocr_tool import detect_and_ocr_slides,slide_extraction_tool

from emotiefflib.facial_analysis import EmotiEffLibRecognizer, get_model_list
from video_analysis.clothing_model import ClothesCLIP

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")

print("OK")

## 2. Model Cache and Paths
Set project-local cache paths and define input/output locations.


In [ ]:
MODEL_CACHE = Path("model_cache")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(MODEL_CACHE)

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
os.environ["PATH"] = str(Path(ffmpeg_exe).parent) + os.pathsep + os.environ.get("PATH", "")

VIDEO_PATH = str(Path("video") / "ppt_video.mp4")
WEIGHTS_DIR = MODEL_CACHE / "weights"
DETECTRON_MODEL = str("/datasets/model_best/model_best.pth")
DETECTRON_CONFIG = str(WEIGHTS_DIR / "my_custom_config.yaml")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model cache:", MODEL_CACHE.resolve())
print("ffmpeg:", ffmpeg_exe)
print("Video:", VIDEO_PATH)
print("Detectron weights:", DETECTRON_MODEL)
print("Detectron config:", DETECTRON_CONFIG)
print("GPU available:", torch.cuda.is_available())

## 3. Helper Utilities
Lightweight helpers for cleanup and safe casting.


In [ ]:
def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Cleanup complete")

def safe_int(val, default=0):
    try:
        return int(val)
    except Exception:
        return default

def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default


## 4. Speech Analysis
Transcribe the audio and compute filler words, intelligibility, noise, and pacing.


In [ ]:
print("Running speech analysis...")
speech_out = analyze_speech_tool(
    video_path=VIDEO_PATH,
    language="en",
    intelligibility_segment_len=30,
)

print("Speech keys:", list(speech_out.keys()))
print("Speech segments:", len(speech_out.get("segments", [])))

force_cleanup()

## 5. Slide Detection (SSIM + OCR)
Detect slide transitions visually, then refine with OCR to merge duplicate segments.


In [ ]:
print("Running slide extraction (SSIM + OCR refine)...")
slides_out = slide_extraction_tool(
    video_path=VIDEO_PATH,
    model_path=DETECTRON_MODEL,
    config_path=DETECTRON_CONFIG,
    ssim_thresh=0.70,
    min_segment_sec=10.0,
    similarity_threshold=0.78,
    min_word_count_for_slide=15,
)

final_slides = slides_out["segments"]
print("Raw segments:", slides_out["raw_count"])
print("Final slides:", slides_out["final_count"])

if final_slides:
    print("Slide 1 OCR preview:")
    print(final_slides[0].get("ocr_text", "")[:400])

force_cleanup()

## 6. Align Audio with Slides
Attach speech segments to each slide window for per-slide recommendations.


In [ ]:
print("Aligning speech segments to slides...")

# Prefer raw segments if available (more metadata + often better text)
speech_segments = speech_out.get("segments_raw") or speech_out.get("segments", [])

final_timeline = []
for slide in final_slides:
    s_start = safe_float(slide.get("start_time", 0.0))
    s_end = safe_float(slide.get("end_time", s_start))
    s_dur = max(0.0, s_end - s_start)

    overlap_text = []
    overlap_speech_sec = 0.0

    for seg in speech_segments:
        a_start = safe_float(seg.get("start", 0.0))
        a_end = safe_float(seg.get("end", a_start))
        latest_start = max(s_start, a_start)
        earliest_end = min(s_end, a_end)

        if earliest_end > latest_start:
            # accumulate overlap duration
            overlap_speech_sec += (earliest_end - latest_start)

            # text (raw segments use "text" too)
            t = (seg.get("text") or "").strip()
            if t:
                overlap_text.append(t)

    spoken_text = " ".join(overlap_text).strip()
    spoken_word_count = len(spoken_text.split())

    final_timeline.append({
        "slide_id": safe_int(slide.get("slide_id")),
        "start_time": s_start,
        "end_time": s_end,
        "duration_sec": round(s_dur, 3),
        "visual_text": slide.get("ocr_text", "") or "",
        "visual_word_count": safe_int(slide.get("ocr_word_count", 0)),
        "spoken_text": spoken_text,
        "spoken_word_count": spoken_word_count,
        "speech_overlap_sec": round(overlap_speech_sec, 3),
        "speech_coverage_ratio": round((overlap_speech_sec / s_dur) if s_dur > 0 else 0.0, 3),
    })

# Handy map (kept compatible with what you had)
idx_to_slide = {
    int(seg["slide_id"]): {
        "slide_content": seg.get("visual_text", ""),
        "audio_content": seg.get("spoken_text", ""),
        "speech_coverage_ratio": seg.get("speech_coverage_ratio", 0.0),
    }
    for seg in final_timeline
}

print("Timeline created:", len(final_timeline))

# Quick debug summary
for seg in final_timeline[:5]:
    print(
        "slide", seg["slide_id"],
        "coverage=", seg["speech_coverage_ratio"],
        "spoken_words=", seg["spoken_word_count"],
        "visual_words=", seg["visual_word_count"],
        "t=", round(seg["start_time"], 2), "-", round(seg["end_time"], 2),
    )


## 7. Face Cache (Shared for Visual Analyses)
Sample frames per slide and cache face crops for downstream models.


In [ ]:
results = {
    "video_info": {"fps": None},
    "segments": final_timeline,
}

cache_out = build_face_cache_tool(
    video_path=VIDEO_PATH,
    segments=results["segments"],
    fps=results.get("video_info", {}).get("fps"),
    per_slide_frames=12,
    batch_size=24,
)

results["video_info"]["fps"] = cache_out["fps"]
results["slide_frame_mapping"] = cache_out["slide_frame_mapping"]
results["face_crops_cache"] = cache_out["face_crops_cache"]
results["face_cache_stats"] = cache_out["stats"]

print("Face cache stats:", results["face_cache_stats"])

## 8. Clothing Analysis 
CLIP-based attire classification; uses aggregated frames across slides.


In [ ]:
clothing_classifier = ClothesCLIP()
print(
    "Using CLIP model for clothing analysis."
    if (clothing_classifier.model and clothing_classifier.processor)
    else "Using fallback clothing analysis."
)

clothing_out = clothing_analysis_tool(
    video_path=VIDEO_PATH,
    slide_frame_mapping=results["slide_frame_mapping"],
    face_crops_cache=results["face_crops_cache"],
    clothing_classifier=clothing_classifier,
    frames_per_slide_max=4,
    min_face_conf=0.55,
)

results["clothing_analysis"] = {
    "is_appropriate": clothing_out["is_appropriate"],
    "detected_attributes": clothing_out["detected_attributes"],
    "recommendation": clothing_out["recommendation"],
    "coverage": clothing_out["coverage"],
}

print("Clothing analysis:", results["clothing_analysis"])

del clothing_classifier
force_cleanup()

## 9. Emotion Analysis
Per-slide emotion distribution and overall dominant emotion.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fer = EmotiEffLibRecognizer(engine="onnx", model_name=get_model_list()[0], device=device)

emotion_out = emotion_analysis_tool(
    video_path=VIDEO_PATH,
    slide_frame_mapping=results["slide_frame_mapping"],
    face_crops_cache=results["face_crops_cache"],
    fer=fer,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=6,
    min_face_conf=0.55,
)

results["emotion_analysis"] = emotion_out
print("Overall emotion stats:", emotion_out.get("overall_stats", {}))

for seg in results["segments"]:
    sid = str(seg["slide_id"])
    slide_summary = emotion_out.get("slide_summaries", {}).get(sid, {})
    seg["dominant_emotion"] = slide_summary.get("dominant_emotion")
    seg["emotion_confidence"] = slide_summary.get("avg_confidence", 0.0)

del fer
force_cleanup()

## 10. Gaze Analysis (Overall)
Track head/gaze direction and summarize eye contact quality.


In [ ]:
from video_analysis.gaze_estimator import MediaPipeGazeDirection
gaze_estimator = MediaPipeGazeDirection()
gaze_out = gaze_analysis_tool(
    VIDEO_PATH,
    results["slide_frame_mapping"],
    results["face_crops_cache"],
    gaze_estimator,
    idx_to_slide,
)

results["gaze_analysis"] = gaze_out
print("Gaze summary:", gaze_out.get("overall_summary", {}))

gaze_estimator.close()

## 11. Gesture Analysis (Overall)
Pose-based body language stats and recommendations.


In [ ]:
import gdown
from video_analysis.gesture_analysis import Gestures

weights_path = Path("yolov8n-pose.pt")
if not weights_path.exists():
    file_id = "1qkEOE92d1we8Mp56I-0NsqTD603hsGoP"
    gdown.download(f"https://drive.google.com/uc?id={file_id}", output=str(weights_path), quiet=False)

gesture_detector = Gestures(str(weights_path))

gesture_out = gestures_analysis_tool(
    video_path=VIDEO_PATH,
    slide_frame_mapping=results["slide_frame_mapping"],
    gesture_detector=gesture_detector,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=6,
)

results["gesture_analysis"] = gesture_out
print("Gesture overall:", gesture_out.get("overall", {}).get("recommendations", {}))

force_cleanup()

## 12. Recommendations (NeMo ReAct)
Recommendations are generated only via NeMo Agent Toolkit (ReAct).
Use the NAT config at `speaker_feedback_nemo/configs/recommendations.yml` to run the agent.
Ensure `OPENROUTER_API_KEY` is set in your environment.

Edit the constraints below to match the talk (type, audience, goal, time limit).


In [ ]:
os.environ["OPENROUTER_API_KEY"] = ""

In [ ]:
from speaker_feedback_nemo.recommendations_runner import (
    generate_recommendations,
    write_markdown_report,
    write_pdf_report,
)

# Update these constraints for your specific presentation.
constraints = {
    "presentation_type": "",
    "audience": "",
    "goal": "",
    "time_limit": "",
}

recommendations = generate_recommendations(
    payload_path="outputs/analysis_payload.json",
    config_path="speaker_feedback_nemo/configs/recommendations.yml",
    constraints=constraints,
    top_k=6,
)

report_path = write_markdown_report(
    recommendations,
    "outputs/recommendations_report.md",
)

# Optional: requires reportlab
# write_pdf_report(recommendations, "outputs/recommendations_report.pdf")

print(recommendations)
print(f"Wrote report to {report_path}")
